# Task 2.1 — Build silver.flight_operations — Complete Operational Timeline

Join `flight_schedule` with `gate_events` to compute operational timestamps:
- `gate_open_ts`, `boarding_start_ts`, `boarding_complete_ts`, `pushback_ts` (= actual departure)
- Derive: `gate_turn_time_mins` (gate_released minus gate_open for prior inbound flight on same gate)
- Derive: `boarding_duration_mins`, `delay_category` (ON_TIME / MINOR / MODERATE / SEVERE)
- UPSERT using MERGE INTO on `flight_id` — refreshed every 15 minutes as new gate events arrive


In [ ]:
# ── Imports & Configuration ────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

DATA_DIR = "/FileStore/airport_analytics_data"
BRONZE_DIR = "/FileStore/delta_lake/bronze"
SILVER_DIR = "/FileStore/delta_lake/silver"

try:
    spark
except NameError:
    spark = (SparkSession.builder
        .appName("Task_2_1_Flight_Operations")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate())


## Step 1 — Read Bronze Tables

In [ ]:
# ── Read Bronze Tables ────────────────────────────────────────────
bronze_flights_df = spark.read.format("delta").load(f"{BRONZE_DIR}/flight_schedule")
bronze_gate_df = spark.read.format("delta").load(f"{BRONZE_DIR}/gate_events")

print(f"Bronze flights: {bronze_flights_df.count()}")
print(f"Bronze gate events: {bronze_gate_df.count()}")
bronze_flights_df.show(3, truncate=False)
bronze_gate_df.show(3, truncate=False)


## Step 2 — Pivot Gate Events to Operational Timestamps per Flight

In [ ]:
# ── Pivot gate events: one row per flight with event timestamps ───
gate_pivoted_df = (bronze_gate_df
    .groupBy("flight_id", "gate_id")
    .pivot("event_type", [
        "GATE_ASSIGNED", "GATE_OPEN", "BOARDING_START",
        "BOARDING_COMPLETE", "PUSHBACK", "GATE_CLOSED", "GATE_RELEASED"
    ])
    .agg(F.first("event_ts"))
)

# Rename pivoted columns for clarity
gate_pivoted_df = (gate_pivoted_df
    .withColumnRenamed("GATE_ASSIGNED", "gate_assigned_ts")
    .withColumnRenamed("GATE_OPEN", "gate_open_ts")
    .withColumnRenamed("BOARDING_START", "boarding_start_ts")
    .withColumnRenamed("BOARDING_COMPLETE", "boarding_complete_ts")
    .withColumnRenamed("PUSHBACK", "pushback_ts")
    .withColumnRenamed("GATE_CLOSED", "gate_closed_ts")
    .withColumnRenamed("GATE_RELEASED", "gate_released_ts")
)

print(f"Pivoted gate events (flights with events): {gate_pivoted_df.count()}")
gate_pivoted_df.show(5, truncate=False)


## Step 3 — Join & Derive Operational Metrics

In [ ]:
# ── Join flight schedule with pivoted gate events ────────────────
flight_ops_df = (bronze_flights_df
    .join(gate_pivoted_df, on="flight_id", how="left")
)

# Derive boarding_duration_mins
flight_ops_df = flight_ops_df.withColumn(
    "boarding_duration_mins",
    F.round(
        (F.unix_timestamp("boarding_complete_ts") -
         F.unix_timestamp("boarding_start_ts")) / 60, 1
    )
)

# Derive actual_departure from pushback_ts
flight_ops_df = flight_ops_df.withColumn(
    "actual_departure",
    F.coalesce("pushback_ts", "scheduled_departure")
)

# Derive delay_category based on delay_minutes
flight_ops_df = flight_ops_df.withColumn(
    "delay_category",
    F.when(F.col("delay_minutes") <= 0, "ON_TIME")
     .when(F.col("delay_minutes") <= 14, "MINOR")
     .when(F.col("delay_minutes") <= 60, "MODERATE")
     .otherwise("SEVERE")
)

# Derive gate_turn_time_mins — time between gate_released of previous
# flight and gate_open of current flight on the same gate
gate_window = Window.partitionBy("gate_id").orderBy("gate_open_ts")

flight_ops_df = flight_ops_df.withColumn(
    "prev_gate_released_ts",
    F.lag("gate_released_ts").over(gate_window)
)
flight_ops_df = flight_ops_df.withColumn(
    "gate_turn_time_mins",
    F.round(
        (F.unix_timestamp("gate_open_ts") -
         F.unix_timestamp("prev_gate_released_ts")) / 60, 1
    )
)

# Add operational date and processing timestamp
flight_ops_df = (flight_ops_df
    .withColumn("operational_date", F.to_date("scheduled_departure"))
    .withColumn("processing_ts", F.current_timestamp())
    .drop("prev_gate_released_ts")
)

print(f"Flight operations rows: {flight_ops_df.count()}")
flight_ops_df.select(
    "flight_id", "flight_number", "delay_minutes", "delay_category",
    "boarding_duration_mins", "gate_turn_time_mins"
).show(10, truncate=False)


## Step 4 — Write to Silver Delta (UPSERT on flight_id)

In [ ]:
# ── Write silver.flight_operations (MERGE / UPSERT) ──────────────
silver_flight_ops_path = f"{SILVER_DIR}/flight_operations"

if DeltaTable.isDeltaTable(spark, silver_flight_ops_path):
    silver_table = DeltaTable.forPath(spark, silver_flight_ops_path)
    silver_table.alias("tgt").merge(
        flight_ops_df.alias("src"),
        "tgt.flight_id = src.flight_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f"UPSERT completed on {silver_flight_ops_path}")
else:
    (flight_ops_df.write
        .format("delta")
        .mode("overwrite")
        .save(silver_flight_ops_path))
    print(f"Initial load to {silver_flight_ops_path}")

# Verify
silver_ops_df = spark.read.format("delta").load(silver_flight_ops_path)
print(f"Silver flight_operations total rows: {silver_ops_df.count()}")
print("\nDelay Category Distribution:")
silver_ops_df.groupBy("delay_category").count().orderBy("count", ascending=False).show()
